# 🎓 Fine-Tune Your Own `Haroony-Physics-AI` Model for Free (Google Colab T4 GPU)

This notebook fine-tunes an open-source **Qwen2.5-3B-Instruct** model on **Harony Physics Clinic's** custom dataset (`data/training/haroony_sft_dataset.jsonl`, built from *Harony Notes 2026 Final* + *Cambridge 0625 Mark Schemes* + *Physics-Only Guardrail refusals*) using **Unsloth QLoRA** on Google Colab's **100% Free T4 GPU**, then exports a quantized `.gguf` file you can run forever with **zero rate limits** via Ollama!

In [ ]:
# Step 1: Install Unsloth (Free Colab T4 GPU)
!pip install -q unsloth datasets trl transformers peft accelerate bitsandbytes

In [ ]:
# Step 2: Load Open-Source Base Model in 4-bit QLoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
)

In [ ]:
# Step 3: Load Haroony Physics SFT Dataset (upload data/training/haroony_sft_dataset.jsonl to Colab)
from datasets import load_dataset

dataset = load_dataset('json', data_files='haroony_sft_dataset.jsonl', split='train')
def format_chat(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(format_chat)
print(f'Loaded {len(dataset)} Harony Physics training examples!')

In [ ]:
# Step 4: Train Haroony Physics AI (~8 minutes on free T4 GPU)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir='haroony_lora_out',
    ),
)
trainer.train()

In [ ]:
# Step 5: Export to GGUF (Q4_K_M) for 100% Free Unlimited Self-Hosting with Ollama!
model.save_pretrained_gguf('haroony_physics_gguf', tokenizer, quantization_method='q4_k_m')